### Pre-processing

In [ ]:
import pandas as pd
from datetime import datetime as dt

rates = [# 0.125 prior
    [dt(2022,3,17), 0.00375],
    [dt(2022,5,4), 0.00875],
    [dt(2022,6,15), 0.01625],
    [dt(2022,7,27), 0.02375],
    [dt(2022,9,21), 0.03125],
]

for y in ['1','2']:
    for q in range(1,13):
        tnr = y+str(q).zfill(2)
        path = "C:\\Users\\Admin\\Downloads\\spxdata{0}\\spx_eod_202{1}{2}"
        read_path = path.format('\\raw', tnr, '.txt')
        data = pd.read_csv(read_path)
    

        d = data[[' [QUOTE_DATE]', ' [UNDERLYING_LAST]', ' [STRIKE]', ' [C_IV]', ' [C_VOLUME]', ' [C_BID]', ' [C_ASK]', ' [P_BID]', ' [P_ASK]', ' [P_IV]', ' [P_VOLUME]', ' [DTE]']]
        for empty in [' [C_VOLUME]', ' [P_VOLUME]', ' [C_BID]', ' [C_ASK]', ' [P_BID]', ' [P_ASK]', ' [C_IV]', ' [P_IV]']:
            d[empty] = pd.to_numeric(d[empty], errors='coerce').fillna(0)
        dict_list = []

        def data_parse(row, lst=dict_list):
            row.update(shared)
            if row['volume'] > 4:
                row.pop('volume', None)
                if row['bid'] and row['bid']<row['ask']:
                    if 0.7 < row['moneyness'] < 1.4:
                        for rate in rates:
                            if dt.strptime(row['qd'], '%Y-%m-%d') > rate[0]:
                                row['r'] = rate[1]
                        dict_list.append(row)
            return dict_list
            

        for idx, row in d.iterrows():
                shared = {'qd': row[' [QUOTE_DATE]'].strip(),'T': (row[' [DTE]']/365.25),'S': row[' [UNDERLYING_LAST]'],'K': row[' [STRIKE]'], 'r': 0.00125}
                px = (row[' [C_BID]']+row[' [C_ASK]'])/2
                crow = {'cp_type':'C', 'iv': float(row[' [C_IV]']),'volume': int(row[' [C_VOLUME]']),'bid': row[' [C_BID]'],'ask': row[' [C_ASK]'], 'px': px, 'moneyness': row[' [UNDERLYING_LAST]']/px}
                px = (row[' [P_BID]']+row[' [P_ASK]'])/2
                prow = {'type':'P', 'iv': float(row[' [P_IV]']),'volume': int(row[' [P_VOLUME]']), 'bid': row[' [P_BID]'],'ask': row[' [P_ASK]'], 'px': px, 'moneyness': row[' [UNDERLYING_LAST]']/px}

                if 0.166 < shared['t'] < 61:
                    crow.update(shared)
                    dict_list = data_parse(crow)
                    prow.update(shared)
                    dict_list = data_parse(prow)

        new_df = pd.DataFrame(dict_list)        
        write_path = path.format('',tnr, '_processed.csv')
        new_df.to_csv(write_path, index=False)
        print(f'{write_path} written, len:{len(new_df)}')

### Processing

In [ ]:
import numpy as np
import pandas as pd
from datetime import timedelta, date
from scipy.optimize import least_squares, minimize, differential_evolution, brentq
from scipy.stats import norm
import logging as log
import matplotlib.pyplot as plt


log.basicConfig(level=log.INFO, format='%(asctime)s %(levelname)s: %(message)s', datefmt='%H:%M:%S')


class HestonCalibrateExperiment:

    def __init__(self, regime, optimiser='LM', logging=True):
        if regime not in ("calm", "stress"):
            raise ValueError(f"regime must be 'calm' or 'stress', got {regime!r}")
        self.regime = regime

        self.market_data = {}
        self.calib_dates = []

        if optimiser not in ("LM", "NM", "DE"):
            raise ValueError(f"optimiser must be one of 'LM', 'NM', 'DE', got {optimiser}")
        self.optimiser = optimiser
        self.logging = logging

        self.history = []

        self.tol_lm = 0.01
        self.tol_nm = 0.1
        self.tol_de = 0.7
        self.bounds = [
            (0.01, 20.0),   # kappa
            (0.001, 1.0),   # theta
            (0.01, 5.0),    # sigma
            (-0.99, 0.99),  # rho
            (0.001, 1.0),   # v0
        ]
        self.load_regime(regime)

    class COSPricer:

        def __init__(self, S, K, T, kappa, theta, sigma, rho, v0, r, option_type):
            self.q = 0.015
            self.mu = r - self.q # drift        
            self.N = 64 # no. of COS terms
            self.price = 0

            self.pricer(S, K, T, kappa, theta, sigma, rho, v0, r, option_type)

        def pricer(self, S, K, T, kappa, theta, sigma, rho, v0, r, option_type):
            """Split C/P pricing as recommended by Fang Osterlee 08"""
            put = self.put_pricer(S, K, T, kappa, theta, sigma, rho, v0, r)
            if option_type not in ['C', 'P']:
                raise ValueError(f"option_type must be 'C' or 'P', got '{option_type}'!")
            if option_type == "P":
                self.price = put
            elif option_type == "C":
                call = put + S * np.exp(-self.q * T) - K * np.exp(-r * T) # eqn 50
                self.price = max(call, 0.0)

        def put_pricer(self, S, K, T, kappa, theta, sigma, rho, v0, r):
            """base P pricing"""

            c1, c2 = self.cumulants(T, kappa, theta, sigma, rho, v0) # eqn 53
            sig =  12 * np.sqrt(np.abs(c2))

            x0 = np.log(S / K) # ruijter osterlee, centre/ mean
            a, b = x0 + c1 - sig, x0 + c1 + sig

            k = np.arange(self.N)
            w = k * np.pi / (b - a) # eqn 34

            cf = self.heston_characteristic(w, T, kappa, theta, sigma, rho, v0) # eqn 34
            phi = cf * np.exp(1j * w * x0) # eqn 33
            Fk = np.real(phi * np.exp(-1j * w * a)) # eqn 19
            Fk[0] = 0.5*Fk[0]  # COS summation convention: halve the k=0 term # eqn 4

            Uk = (2.0 / (b - a)) * (-self.put_chi(k, a, b) + self.put_psi(k, a, b)) # eqn 29
            Vk = K * Uk

            price = np.exp(-r * T) * np.sum(Fk * Vk) # eqn 30
            return max(price, 0)        
        

        def cumulants(self, T, kappa, theta, sigma, rho, v0):
            """calculate cumulants 1 and 2"""

            c1 = self.mu * T + (1 - np.exp(-kappa * T)) * (theta - v0) / (2 * kappa) - 0.5 * theta * T

            A = (v0 / (4 * kappa**3)) * (
                4 * kappa**2 * (1 + (rho * sigma * T - 1) * np.exp(-kappa * T))
                + kappa * (4 * rho * sigma * (np.exp(-kappa * T) - 1) - 2 * sigma**2 * T * np.exp(-kappa * T))
                + sigma**2 * (1 - np.exp(-2 * kappa * T)))
            B = (theta / (8 * kappa**3)) * (
                8 * kappa**3 * T
                - 8 * kappa**2 * (1 + rho * sigma * T + (rho * sigma * T - 1) * np.exp(-kappa * T))
                + 2 * kappa * ((1 + 2 * np.exp(-kappa * T)) * sigma**2 * T + 8 * (1 - np.exp(-kappa * T)) * rho * sigma)
                + sigma**2 * (np.exp(-2 * kappa * T) + 4 * np.exp(-kappa * T) - 5))
            c2 = A + B

            return c1, c2

        def heston_characteristic(self, w, T, kappa, theta, sigma, rho, v0):
            """calculate heston characteristic function, stable Albrecher style"""

            d = np.sqrt((rho * sigma * 1j * w - kappa)**2 + sigma**2 * (w**2 + 1j * w))
            g = (kappa - rho * sigma * 1j * w - d) / (kappa - rho * sigma * 1j * w + d)

            D = ((kappa - rho * sigma * 1j * w - d) / sigma**2) * ((1 - np.exp(-d * T)) / (1 - g * np.exp(-d * T)))
            C = (kappa * theta / sigma**2) * ((kappa - rho * sigma * 1j * w - d) * T - 2 * np.log((1 - g * np.exp(-d * T)) / (1 - g)))

            return np.exp(1j * w * self.mu * T + C + D * v0)

        def put_chi(self, k, a, b): # eqn 22
            '''calculate chi given c = a, d = 0'''
            kpi = k * np.pi / (b - a)
            chi = (np.cos(kpi * a) - np.exp(a) - kpi * np.sin(kpi * a)) / (1.0 + kpi**2)
            return chi

        def put_psi(self, k, a, b): # eqn 23
            '''calculate psi given c = a, d = 0'''
            kpi = k * np.pi / (b - a)
            psi = np.empty_like(kpi, dtype=float)
            nonzero = k != 0 # circumvent 0 division
            psi[nonzero] = -np.sin(kpi[nonzero] * a) / kpi[nonzero]
            psi[~nonzero] = -a
            return psi


    ########### load and clean spx data #########

    def load_market_data(self, yr, mth, target_date=None, params_dict=None):
        """"read and parse option chain csvs"""
    
        def read_csv(yr, mth):
            if yr not in [21, 22] or mth not in range(1, 13):
                raise ValueError("year must be 21 or 22 and mth must be 1-12!")
            path = f"C:\\Users\\Admin\\Downloads\\spxdata\\spx_eod_20{str(yr)+str(mth).zfill(2)}_processed.csv"
            try:
                df = pd.read_csv(path)
                log.info(f"{path} loaded")
            except:
                raise FileNotFoundError(f"Can\'t find {path} :(")
            return df

        def find_second_wed(start, end):
            d = date(start.year, start.month, 1)
            while d.weekday() != 2:
                d += timedelta(days=1)
            second_wed = d + timedelta(weeks=1)
            if start <= second_wed <= end:
                return second_wed
            return None

        def clean_data(df):
            moneyness = df['Asset Price'] / df['Strike']
            df = df[(moneyness >= 0.8) & (moneyness <= 1.2)]
            iv_idx = df[df['Implied Vol'] <= 0.0].index
            for idx in iv_idx:
                row = df.loc[idx]
                iv = self.vol_finder(row['Asset Price'], row['Strike'], row['Maturity'],
                                row['Option Type'], row['Price'], row['Fed Rate'])
                if pd.isna(iv) or iv > 1.25:
                    df.drop(idx, inplace=True)
                else:
                    df.at[idx, 'Implied Vol'] = iv
            return df

        raw = read_csv(yr, mth)

        ### out of sample code start ###
        if target_date is not None:
            day_df = raw[raw['Quote Date'] == str(target_date)]
            if len(day_df) == 0:
                raise ValueError(f"No data for {target_date}")
            quotes = clean_data(day_df)
            if params_dict is not None:
                p = list(params_dict.values())  # dict insertion order: kappa,theta,sigma,rho,v0
                resid = self.model_ivs(p, quotes)
                return float(np.sqrt(np.nanmean(resid**2)))
            return quotes
        ### out of sample code end ###

        
        start = date.strptime(raw.iloc[0]['Quote Date'], '%Y-%m-%d')
        end = date.strptime(raw.iloc[-1]['Quote Date'], '%Y-%m-%d')
        calib_date = find_second_wed(start, end)

        if calib_date is None:
            log.warning(f"No calibration date found for {yr}-{mth}")
            return

        day_df = raw[raw['Quote Date'] == str(calib_date)]
        if len(day_df) == 0:  # 11 May 22 empty... contingency
            fallback = calib_date - timedelta(days=1)
            log.warning(f"No rows for {calib_date}, trying {fallback} instead")
            day_df = raw[raw['Quote Date'] == str(fallback)]
            if len(day_df) > 0:
                calib_date = fallback

        quotes = clean_data(day_df)
        self.calib_dates.append(calib_date)
        self.market_data[calib_date] = quotes

    def load_regime(self, regime):
        year_months = [(21, m) for m in range(1, 11)] if regime == "calm" else [(22, m) for m in range(1, 11)]
        for yr, mth in year_months:
            self.load_market_data(yr, mth)
    
    ########## iv calcs ########

    def vol_finder(self, S, K, T, option_type, price, r):
        """back out vol"""

        def bsm_price(S, K, T, option_type, vol):
            """use bsm and brent to calculate price from option inputs"""
            q = 0.015
            d1 = (np.log(S / K) + (r - q + 0.5 * vol**2) * T) / (vol * np.sqrt(T))
            d2 = d1 - vol * np.sqrt(T)
            if option_type == "C":
                return S * np.exp(-q * T) * norm.cdf( d1) - K * np.exp(-r * T) * norm.cdf( d2)
            return    -S * np.exp(-q * T) * norm.cdf(-d1) + K * np.exp(-r * T) * norm.cdf(-d2) 

        f = lambda vol: bsm_price(S, K, T, option_type, vol) - price
        try:
            return brentq(f, 1e-6, 5.0, xtol=1e-8)
        except ValueError:
            print(f'nan vol_finder: S:{S}, K:{K}, px:{price:.3f}, mness:{S/K:.3f}, T:{T:.3f}')
            return np.nan

    def model_ivs(self, params, market_data):
        """model implied vol"""
        kappa0, theta0, sigma0, rho0, v0 = params
        model_ivs = np.empty(len(market_data))
        for idx, row in enumerate(market_data.itertuples(index=False)):
            S, K, T, r, option_type = (row.S, row.K, row.T, row.r, row.cp_type)
            try:
                cosp = self.COSPricer(S, K, T, kappa0, theta0, sigma0, rho0, v0, r, option_type)
                cos_price = cosp.price
                model_ivs[idx] = self.vol_finder(S, K, T, option_type, cos_price, r)
            except:
                print(row)
                model_ivs[idx] = np.nan
        market_ivs = market_data.iv.to_numpy()
        return model_ivs - market_ivs

    def iv_errors(self, params, market_data, log_file):
        """Vector of model_iv - market_iv, for LM"""
        resid = self.model_ivs(params, market_data)
        n_nan = np.isnan(resid).sum()
        mae = np.nanmean(np.abs(resid))
        if log_file:
            log_file.write(f"params={params}, n_failed={n_nan}/{len(resid)}, mae={mae:.6f}\n")
        print(f"params={params}, n_failed={n_nan}/{len(resid)}, mae={mae:.6f}")
        return np.nan_to_num(resid, nan=5.0)

    def iv_rmse(self, params, market_data,log_file):
        """Scalar iv rmse, for with NM and DE"""
        resid = self.model_ivs(params, market_data)
        n_nan = np.isnan(resid).sum()
        penalised = np.nan_to_num(resid, nan=5.0)
        rmse = float(np.sqrt(np.mean(penalised**2)))
        if log_file:
            log_file.write(f"params={params}, n_failed={n_nan}/{len(resid)}, rmse={rmse:.6f}\n")
        print(f"params={params}, n_failed={n_nan}/{len(resid)}, rmse={rmse:.6f}")
        return rmse

    ########### calibration ###########

    def start_params(self, market_data):
            """implement starting point heuristics"""
            moneyness_dist = (market_data["S"] / market_data["K"] - 1).abs()
            atm = market_data.loc[moneyness_dist.nsmallest(10).index]

            kappa_guess = 2.75 / market_data["T"].min()
            sigma_guess = 0.5
            rho_guess = -0.5
            v0_guess = float(atm["iv"].mean() ** 2)
            theta_guess = v0_guess
            print(f'Starting: K:{kappa_guess:.3f}, sig:{sigma_guess:.3f}, rho:{rho_guess:.3f}, v0:{v0_guess:.3f}, the:{theta_guess:.3f}')

            return np.array([kappa_guess, theta_guess, sigma_guess, rho_guess, v0_guess])
    
    def calibrate(self, market_data):
            """top level calibrate"""
            start_params = self.start_params(market_data)
            print(("params=    kappa    ,    theta    ,    sigma    ,    rho    ,    v0    "))
            log_file = open(f"C:\\Users\\Admin\\Downloads\\spxdata\\logs_{self.calib_dates[0]}.txt", "a", buffering=1)
            if self.optimiser == "LM":
                result = self.calibrate_lm(market_data, start_params, log_file)
            elif self.optimiser == "NM":
                result = self.calibrate_nm(market_data, start_params, log_file)
            else:  # "DE"
                result = self.calibrate_de(market_data, log_file)
    
            self.history.append(result)
            if self.logging:
                print(f"[{self.optimiser}] IV RMSE = {result['iv_rmse']:.6f} ({result['n_evaluations']} evaluations)")
            return result
    
    def calibrate_lm(self, market_data, start_params, log_file):
        res = least_squares(self.iv_errors, start_params, args=(market_data, log_file), method="lm",
                            xtol=self.tol_lm, ftol=self.tol_lm)
        return self.return_results(res.x, market_data, res, log_file, n_evals=res.nfev)

    def calibrate_nm(self, market_data, start_params, log_file):
        res = minimize(self.iv_rmse, start_params, args=(market_data, log_file), method="Nelder-Mead",
                        bounds=self.bounds, options={"xatol":self.tol_nm, "fatol": self.tol_nm, "maxiter": 200})
        return self.return_results(res.x, market_data, res, log_file, n_evals=res.nfev)

    def calibrate_de(self, market_data, log_file):
        res = differential_evolution(self.iv_rmse, self.bounds, args=(market_data, log_file),
                                     popsize=8, tol=self.tol_de, rng=42, maxiter=50)
        return self.return_results(res.x, market_data, res, log_file, n_evals=res.nfev)
        
    def return_results(self, res, market_data, raw_result, log_file, n_evals):
        """parse results"""
        model_iv = self.model_ivs(res, market_data) + market_data['iv'].to_numpy()
        res= {
            "optimiser": self.optimiser,
            "params": dict(zip(("kappa", "theta", "sigma", "rho", "v0"), res)),
            "iv_rmse": self.iv_rmse(res, market_data, None),
            "model_iv": model_iv,
            "n_evaluations": n_evals,
            "converged": getattr(raw_result, "success", None),
            "raw_result": raw_result,
        }
        log_file.write(repr(res))
        return res

    def calibrate_surface(self, max_dates=None):
        """calibrate several dates at once"""
        results = []
        for i, d in enumerate(self.calib_dates):
            if max_dates is not None and i >= max_dates:
                break
            log.info(f'\n\n\n\n*************** Begin calibration for {d} ***************')
            result = self.calibrate(self.market_data[d])
            result["date"] = d
            results.append(result)
        self.history.extend(results)
        return results

    ############## visuals ###############

    def plot_surfaces(self, calib_date):
        """plot iv surface"""
        if calib_date not in self.market_data:
            raise ValueError(f"No market data loaded for {calib_date}")

        market_data = self.market_data[calib_date]
        if market_data is None or len(market_data) == 0:
            raise ValueError(f"Market data for {calib_date} is empty")

        matches = [r for r in self.history if r.get("date") == calib_date]
        if not matches:
            raise ValueError(f"No calibration result found for {calib_date}... has calibrate_panel() run yet?")
        result = matches[-1]  # most recent, in case the same date was calibrated more than once

        if "model_iv" not in result:
            raise ValueError(f"Result for {calib_date} has no 'model_iv', was it produced by return_results()?")

        error = result['model_iv'] - market_data['iv'].to_numpy()

        fig = plt.figure(figsize=(18, 5))
        fig.suptitle(f"IV Surfaces: {str(calib_date)}")
        for i, (z, title, cmap) in enumerate([
            (market_data['iv'], 'Market IV', 'viridis'),
            (result['model_iv'], f"{result['optimiser']} Model IV", 'viridis'),
            (error, 'Model - Market Error', 'coolwarm'),
        ]):
            ax = fig.add_subplot(1, 3, i+1, projection='3d')
            ax.plot_trisurf(market_data['K'], market_data['T'], z, cmap=cmap)
            ax.set_title(title)
            ax.set_xlabel('Strike')
            ax.set_ylabel('Maturity')
            ax.set_zlabel('IV')

        plt.tight_layout()
        return fig

        

### Post-Analysis

In [ ]:
import re
import glob
import os
import pandas as pd

FIELD_PATTERNS = {
    "optimiser":     r"'optimiser':\s*'(\w+)'",
    "kappa":         r"'kappa':\s*np\.float64\(([-\d.eE+]+)\)",
    "theta":         r"'theta':\s*np\.float64\(([-\d.eE+]+)\)",
    "sigma":         r"'sigma':\s*np\.float64\(([-\d.eE+]+)\)",
    "rho":           r"'rho':\s*np\.float64\(([-\d.eE+]+)\)",
    "v0":            r"'v0':\s*np\.float64\(([-\d.eE+]+)\)",
    "iv_rmse":       r"'iv_rmse':\s*([-\d.eE+]+)",
    "n_evaluations": r"'n_evaluations':\s*(\d+)",
    "converged":     r"'converged':\s*(True|False)",
    "date":          r"'date':\s*datetime\.date\((\d+),\s*(\d+),\s*(\d+)\)",
    "optimality":    r"optimality:\s*([-\d.eE+]+)", # lm only
    "nit":           r"\bnit:\s*(\d+)",
}


def parse_log_file(path):
    with open(path) as f:
        text = f.read()

    records = re.split(r"(?=\{'optimiser':)", text)
    records = [r for r in records if "'optimiser':" in r]

    parsed = []
    for rec in records:
        matches = {k: re.search(p, rec) for k, p in FIELD_PATTERNS.items()}

        if not (matches["optimiser"] and matches["kappa"] and matches["date"]):
            print(f"  WARNING: skipped an incomplete record in {os.path.basename(path)}")
            continue

        d = matches["date"]

        parsed.append({
            "optimiser": matches["optimiser"].group(1),
            "kappa": float(matches["kappa"].group(1)),
            "theta": float(matches["theta"].group(1)),
            "sigma": float(matches["sigma"].group(1)),
            "rho": float(matches["rho"].group(1)),
            "v0": float(matches["v0"].group(1)),
            "iv_rmse": float(matches["iv_rmse"].group(1)),
            "n_evaluations": int(matches["n_evaluations"].group(1)) if matches["n_evaluations"] else None,
            "converged": (matches["converged"].group(1) == "True") if matches["converged"] else None,
            "date": f"{d.group(1)}-{int(d.group(2)):02d}-{int(d.group(3)):02d}",
            "optimality": float(matches["optimality"].group(1)) if matches["optimality"] else None,
            "nit": int(matches["nit"].group(1)) if matches["nit"] else None,
            "source_file": os.path.basename(path),
        })
    return parsed


def load_all_logs(folder):
    rows = []
    for path in glob.glob(os.path.join(folder, "logs_*")):
        recs = parse_log_file(path)
        rows.extend(recs)

    df = pd.DataFrame(rows)
    df["date"] = pd.to_datetime(df["date"]).dt.date
    df["regime"] = df["date"].apply(lambda d: "calm" if d.year == 2021 else "stress")
    return df


df = load_all_logs(r"C:\Users\Admin\OneDrive\Documents\logs")
print(df.groupby(["optimiser", "regime"]).size())

optimiser  regime
NM         calm      10
dtype: int64


#### 1. Parameter stability

In [15]:
summary = df.groupby(["optimiser", "regime"]).agg(
    kappa_mean=("kappa", "mean"), kappa_std=("kappa", "std"),
    theta_mean=("theta", "mean"), theta_std=("theta", "std"),
    sigma_mean=("sigma", "mean"), sigma_std=("sigma", "std"),
    rho_mean=("rho", "mean"), rho_std=("rho", "std"),
    v0_mean=("v0", "mean"), v0_std=("v0", "std"),
    rmse_mean=("iv_rmse", "mean"), rmse_std=("iv_rmse", "std"),
).reset_index()

summary = summary.sort_values(["optimiser", "regime"]).reset_index(drop=True)
summary

,optimiser,regime,kappa_mean,kappa_std,theta_mean,theta_std,sigma_mean,sigma_std,rho_mean,rho_std,v0_mean,v0_std,rmse_mean,rmse_std
0,NM,calm,2.261575,1.904041,0.085741,0.031442,0.974026,0.430376,-0.775438,0.114465,0.034623,0.014317,0.011342,0.005157


#### 2. Out-of-Sample RMSE

In [16]:
hce = HestonCalibrateExperiment('calm', 'NM')

14:38:41 INFO: C:\Users\Admin\Downloads\spxdata\spx_eod_202101_processed.csv loaded
14:38:41 INFO: C:\Users\Admin\Downloads\spxdata\spx_eod_202102_processed.csv loaded
14:38:41 INFO: C:\Users\Admin\Downloads\spxdata\spx_eod_202103_processed.csv loaded
14:38:41 INFO: C:\Users\Admin\Downloads\spxdata\spx_eod_202104_processed.csv loaded
14:38:42 INFO: C:\Users\Admin\Downloads\spxdata\spx_eod_202105_processed.csv loaded
14:38:42 INFO: C:\Users\Admin\Downloads\spxdata\spx_eod_202106_processed.csv loaded
14:38:42 INFO: C:\Users\Admin\Downloads\spxdata\spx_eod_202107_processed.csv loaded
14:38:42 INFO: C:\Users\Admin\Downloads\spxdata\spx_eod_202108_processed.csv loaded
14:38:42 INFO: C:\Users\Admin\Downloads\spxdata\spx_eod_202109_processed.csv loaded
14:38:42 INFO: C:\Users\Admin\Downloads\spxdata\spx_eod_202110_processed.csv loaded


In [ ]:
from datetime import date as dt_date, timedelta

weds = {"calm":   {"wed": dt_date(2021, 3, 10), "yr": 21, "mth": 3}}
# weds = {"calm":   {"wed": dt_date(2021, 5, 12), "yr": 21, "mth": 5}}
# weds = {"calm":   {"wed": dt_date(2021, 8, 11), "yr": 21, "mth": 8}}

oos_results = []

for regime, info in weds.items():
    wed = info["wed"]
    target = wed + timedelta(days=7)  # +1 week = t+5 trading days

    for opt in ("LM", "NM", "DE"):
        row = df[(df.optimiser == opt) & (df.regime == regime) & (df.date == wed)]
        if len(row) == 0:
            print(f"No calibrated result found for {opt}/{regime}/{wed} — skipping")
            continue
        row = row.iloc[0]
        params = {'kappa': row.kappa, 'theta': row.theta, 'sigma': row.sigma, 'rho': row.rho, 'v0': row.v0}

        hce = HestonCalibrateExperiment(regime, opt)
        try:
            oos_rmse = hce.load_market_data(info["yr"], info["mth"], target_date=target, params_dict=params)
        except ValueError as e:
            print(f"{opt}/{regime}: {e}")
            continue

        in_sample_rmse = row.iv_rmse
        oos_results.append({
            "optimiser": opt, "regime": regime,
            "wed_date": wed, "target_date": target,
            "in_sample_rmse": in_sample_rmse, "out_of_sample_rmse": oos_rmse,
            "degradation": oos_rmse - in_sample_rmse,
        })
        print(f"{opt}/{regime}: in-sample={in_sample_rmse:.5f}, out-of-sample={oos_rmse:.5f}, "
              f"degradation={oos_rmse - in_sample_rmse:+.5f}")

oos_df = pd.DataFrame(oos_results)
oos_df

No calibrated result found for LM/calm/2021-08-11 — skipping


14:39:57 INFO: C:\Users\Admin\Downloads\spxdata\spx_eod_202101_processed.csv loaded
14:39:58 INFO: C:\Users\Admin\Downloads\spxdata\spx_eod_202102_processed.csv loaded
14:39:58 INFO: C:\Users\Admin\Downloads\spxdata\spx_eod_202103_processed.csv loaded
14:39:58 INFO: C:\Users\Admin\Downloads\spxdata\spx_eod_202104_processed.csv loaded
14:39:58 INFO: C:\Users\Admin\Downloads\spxdata\spx_eod_202105_processed.csv loaded
14:39:59 INFO: C:\Users\Admin\Downloads\spxdata\spx_eod_202106_processed.csv loaded
14:39:59 INFO: C:\Users\Admin\Downloads\spxdata\spx_eod_202107_processed.csv loaded
14:39:59 INFO: C:\Users\Admin\Downloads\spxdata\spx_eod_202108_processed.csv loaded
14:39:59 INFO: C:\Users\Admin\Downloads\spxdata\spx_eod_202109_processed.csv loaded
14:40:00 INFO: C:\Users\Admin\Downloads\spxdata\spx_eod_202110_processed.csv loaded
14:40:00 INFO: C:\Users\Admin\Downloads\spxdata\spx_eod_202108_processed.csv loaded


NM/calm: in-sample=0.01781, out-of-sample=0.03082, degradation=+0.01301
No calibrated result found for DE/calm/2021-08-11 — skipping


,optimiser,regime,anchor_date,target_date,in_sample_rmse,out_of_sample_rmse,degradation
0,NM,calm,2021-08-11,2021-08-18,0.017811,0.030818,0.013008


#### 3. Compute cost

In [20]:
compute_summary = df.groupby(["optimiser", "regime"]).agg(
    n_evals_mean=("n_evaluations", "mean"),
    n_evals_std=("n_evaluations", "std"),
    n_evals_min=("n_evaluations", "min"),
    n_evals_max=("n_evaluations", "max"),
).reset_index()

compute_summary = compute_summary.sort_values(["optimiser", "regime"]).reset_index(drop=True)
compute_summary

,optimiser,regime,n_evals_mean,n_evals_std,n_evals_min,n_evals_max
0,NM,calm,194.7,135.570441,119,573


#### Appendix

In [21]:
for (opt, reg), group in df.groupby(['optimiser', 'regime']):
    print(f"\n{opt} {reg}")
    print(group[['date','kappa','theta','sigma','rho','v0','iv_rmse']].to_string(index=False))


NM calm
      date    kappa    theta    sigma       rho       v0  iv_rmse
2021-01-13 1.219781 0.112740 1.089639 -0.805878 0.048782 0.006580
2021-02-10 0.937025 0.116760 0.971600 -0.508399 0.052308 0.018525
2021-03-10 1.062460 0.077501 0.604101 -0.799298 0.049195 0.013187
2021-04-14 1.825266 0.079484 0.938135 -0.724070 0.024395 0.010466
2021-05-12 0.678939 0.094476 0.624356 -0.876670 0.052761 0.012734
2021-06-09 6.746158 0.054695 2.041643 -0.737084 0.026843 0.005448
2021-07-14 4.079298 0.046467 0.671389 -0.950098 0.019216 0.016507
2021-08-11 2.981222 0.050087 0.609910 -0.781417 0.017888 0.017811
2021-09-08 0.955930 0.142621 1.102462 -0.786710 0.026005 0.006499
2021-10-13 2.129675 0.082572 1.087028 -0.784755 0.028837 0.005660
